# Repaso de Python para el bloque 4

Este notebook repasa **las piezas de Python que aparecen en todo el código del bloque**, con ejemplos de
Tostadero Retiro, la distribuidora de café que usaremos como caso.

**Cómo usarlo**
- Arriba a la derecha, elige el kernel del entorno virtual (`.venv`).
- Ejecuta cada celda con `Shift+Enter`.
- En las celdas de ejercicio, sustituye los `...` por tu código. La celda siguiente comprueba tu solución:
  si no sale ningún error, está bien.

Si ya te manejas con Python, salta directamente al **reto final**.

## 1. Diccionarios y listas

Las APIs de los modelos devuelven datos con forma de **diccionario**: pares clave → valor.
Una lista de diccionarios es la forma típica de representar varias filas (productos, pedidos, mensajes…).

In [ ]:
product = {"sku": "CAF-001", "name": "Café Etiopía 1 kg", "price": 24.50, "stock": 120}

print(product["name"])          # read a value
product["stock"] -= 10          # update a value
print(product.get("discount"))  # .get() returns None instead of failing if the key is missing

products = [
    {"sku": "CAF-001", "name": "Café Etiopía 1 kg", "price": 24.50, "stock": 110},
    {"sku": "CAF-003", "name": "Café descafeinado 1 kg", "price": 19.80, "stock": 45},
    {"sku": "MOL-020", "name": "Molino on-demand", "price": 640.00, "stock": 9},
]
for p in products:
    print(f"{p['sku']}: {p['stock']} unidades")

**Ejercicio 1.** Usando una *list comprehension*, crea `low_stock` con los **nombres** de los productos
con menos de 50 unidades en stock.

In [ ]:
low_stock = ...
low_stock

In [ ]:
assert low_stock == ["Café descafeinado 1 kg", "Molino on-demand"]
print("✅ Correcto")

## 2. JSON

JSON es el formato de texto con el que viajan los datos por HTTP. En Python:
- `json.loads(text)` convierte texto JSON en diccionarios y listas.
- `json.dumps(data)` hace lo contrario.

In [ ]:
import json

response_text = '''
{
  "id": "resp_123",
  "output_text": "Es un correo mixto: incidencia y pedido.",
  "usage": {"input_tokens": 78, "output_tokens": 61}
}
'''
response = json.loads(response_text)
print(type(response))
print(response["output_text"])
print(json.dumps({"sku": "CAF-001", "qty": 4}, ensure_ascii=False))

**Ejercicio 2.** A partir de `response`, calcula `total_tokens`: la suma de los tokens de entrada y de salida.

In [ ]:
total_tokens = ...
total_tokens

In [ ]:
assert total_tokens == 139
print("✅ Correcto")

## 3. Funciones y f-strings

Una función recibe parámetros (algunos con valor por defecto) y devuelve un resultado.
Los *f-strings* (`f"..."`) insertan valores en un texto; `:.2f` formatea con dos decimales.

In [ ]:
def line_total(qty: int, price: float, discount: float = 0.0) -> float:
    """Total of an order line, applying an optional discount between 0 and 1."""
    return qty * price * (1 - discount)

total = line_total(4, 24.50)
print(f"Total: {total:.2f} €")
print(f"Con un 10 % de descuento: {line_total(4, 24.50, discount=0.10):.2f} €")

**Ejercicio 3.** Escribe `cost_usd(input_tokens, output_tokens, price_in, price_out)`, que devuelva el coste
en dólares de una llamada a un LLM. Los precios vienen **por millón de tokens**.
(Es justo el cálculo que haremos en la sesión 1.)

In [ ]:
def cost_usd(input_tokens: int, output_tokens: int, price_in: float, price_out: float) -> float:
    ...

In [ ]:
assert round(cost_usd(300_000, 150_000, 0.40, 1.60), 2) == 0.36
print("✅ Correcto")

## 4. `@dataclass` y type hints

Una **dataclass** es una clase pensada para guardar datos: Python genera el constructor por ti.
Las anotaciones de tipo (`name: str`) documentan qué espera cada campo y ayudan al editor (y a Copilot).
En el kit, la respuesta de un modelo se guarda en una dataclass `Respuesta`.

In [ ]:
from dataclasses import dataclass


@dataclass
class Product:
    sku: str
    name: str
    price: float
    stock: int

    @property
    def stock_value(self) -> float:
        return self.price * self.stock


grinder = Product("MOL-020", "Molino on-demand", 640.00, 9)
print(grinder)
print(f"Valor en almacén: {grinder.stock_value:.2f} €")

**Ejercicio 4.** Crea una dataclass `OrderLine` con los campos `sku` (str), `qty` (int) y `unit_price` (float),
y una propiedad `total` que devuelva `qty * unit_price`.

In [ ]:
@dataclass
class OrderLine:
    ...

In [ ]:
line = OrderLine("CAF-002", 10, 16.90)
assert round(line.total, 2) == 169.00
print("✅ Correcto")

## 5. Excepciones: `raise` y `try/except`

Cuando algo no puede hacerse, una función **lanza** una excepción (`raise`). Quien la llama decide
qué hacer con ella (`try/except`). En el bloque lo usaremos para tratar errores de las APIs
(claves inválidas, límites de uso…) y para devolver mensajes útiles al modelo.

In [ ]:
def check_qty(qty: int) -> int:
    if qty <= 0:
        raise ValueError(f"La cantidad debe ser positiva (recibido: {qty})")
    return qty

try:
    check_qty(-3)
except ValueError as error:
    print(f"No se puede crear la línea: {error}")

**Ejercicio 5.** Escribe `find_product(sku)` que busque en la lista `products` (sección 1) y devuelva el diccionario
del producto. Si no existe, debe lanzar `KeyError` con el mensaje `"No existe el producto <sku>"`.

In [ ]:
def find_product(sku: str) -> dict:
    ...

In [ ]:
assert find_product("MOL-020")["name"] == "Molino on-demand"
try:
    find_product("XXX-999")
    raise AssertionError("Debería haber lanzado KeyError")
except KeyError as error:
    assert "XXX-999" in str(error)
print("✅ Correcto")

## 6. Variables de entorno y `.env`

Las claves de API **nunca** se escriben en el código. Se leen de variables de entorno, que en el bloque
cargamos desde el fichero `.env` con `python-dotenv`.

Fíjate en que nunca imprimimos la clave: solo comprobamos si está.

In [ ]:
import os

try:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))  # reads .env from the current folder or a parent one
except ImportError:
    print("python-dotenv no está instalado: activa el entorno virtual del kit.")

for name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]:
    status = "configurada" if os.getenv(name) else "falta"
    print(f"{name}: {status}")

**Ejercicio 6.** Escribe `get_model(provider)` que devuelva el nombre de modelo guardado en la variable de entorno
`<PROVIDER>_MODEL` (por ejemplo `OPENAI_MODEL` para `"openai"`). Si la variable no existe, debe devolver
`"modelo-por-defecto"`.

In [ ]:
def get_model(provider: str) -> str:
    ...

In [ ]:
os.environ["DEMO_MODEL"] = "demo-mini"
assert get_model("demo") == "demo-mini"
assert get_model("inexistente") == "modelo-por-defecto"
print("✅ Correcto")

## 7. `with` y `sqlite3`

`with` abre un recurso y garantiza que, al salir del bloque, se hace la limpieza **aunque haya un error**:
con un fichero, cerrarlo; con una conexión de `sqlite3`, confirmar los cambios (*commit*) o deshacerlos (*rollback*).

El kit guarda los datos de Tostadero Retiro en **SQLite**, una base de datos que es un único fichero (`tostadero.db`).
El módulo `sqlite3` viene con Python. Aquí usamos una base de datos **en memoria** (`":memory:"`), así que no se
escribe nada en disco.

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as folder:  # another context manager: deletes the folder at the end
    path = os.path.join(folder, "notes.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("Pedido 1004: 4 x CAF-001\n")
    with open(path, encoding="utf-8") as f:
        print(f.read().strip())
    print("¿Cerrado al salir del with?", f.closed)

In [ ]:
import sqlite3

con = sqlite3.connect(":memory:")  # an in-memory database: nothing is written to disk
con.row_factory = sqlite3.Row      # rows that can be turned into dictionaries

with con:  # commit at the end of the block (or rollback if there is an error)
    con.execute("CREATE TABLE products (sku TEXT PRIMARY KEY, name TEXT, price REAL, stock INTEGER)")
    con.executemany("INSERT INTO products VALUES (:sku, :name, :price, :stock)", products)

# Each ? is filled with a value from the tuple, in order. Never paste values into the SQL by hand.
rows = con.execute("SELECT sku, price FROM products WHERE price < ?", (100,)).fetchall()
print([dict(r) for r in rows])

**Ejercicio 7.** Escribe `search_products(con, text)`, que devuelva una **lista de diccionarios** con `sku`, `name`
y `stock` de los productos cuyo nombre contenga `text`. Usa `LIKE ?` con el valor `f"%{text}%"` (así lo hace
`stock_producto` en `herramientas.py`) y una *list comprehension* para convertir las filas.

In [ ]:
def search_products(con: sqlite3.Connection, text: str) -> list[dict]:
    ...

In [ ]:
result = search_products(con, "café")
assert sorted(p["sku"] for p in result) == ["CAF-001", "CAF-003"]
assert set(result[0]) == {"sku", "name", "stock"}
assert search_products(con, "tetera") == []
print("✅ Correcto")

## 8. Módulos: un script usa otro

Cada fichero `.py` es un **módulo**. Los scripts del kit se importan unos a otros
(`from herramientas import FUNCIONES`, `from llm_client import LLM`…). Python los encuentra porque están
**en la misma carpeta**: por eso todo el bloque va en una sola carpeta, `bloque4`, con un único `.venv` y un único `.env`.

La variable `__name__` vale `"__main__"` en el fichero que se ejecuta con `python fichero.py`, y el nombre del módulo
cuando se importa. Con `if __name__ == "__main__":` un fichero sirve a la vez como script y como librería.

La celda siguiente crea un módulo de prueba en una carpeta temporal, lo **importa** y luego lo **ejecuta** como script.

In [ ]:
import contextlib
import importlib
import io
import subprocess
import sys

demo_folder = tempfile.mkdtemp()  # a temporary folder for the example modules
sys.path.insert(0, demo_folder)   # so that import can find them (like files in the same folder)


def write_module(name: str, source: str) -> str:
    path = os.path.join(demo_folder, f"{name}.py")
    with open(path, "w", encoding="utf-8") as f:
        f.write(source)
    return path


greeting_path = write_module("greeting", """
def greet(name):
    return f"Hola, {name}"

print("Cargando greeting, __name__ =", __name__)

if __name__ == "__main__":
    print(greet("Bar La Esquina"))
""")

import greeting  # imported: the if block does not run

print(greeting.greet("Hotel Prado Suites"))
print("--- ahora como script ---")
print(subprocess.run([sys.executable, greeting_path], capture_output=True, text=True).stdout)

**Ejercicio 8.** Completa el texto de un módulo `pricing.py` para que:
1. Tenga una función `with_vat(amount)` que devuelva el importe con IVA (21 %) redondeado a 2 decimales.
2. Al **ejecutarlo** como script imprima `98.0 € + IVA = 118.58 €`.
3. Al **importarlo** no imprima nada.

La celda de comprobación lo escribe en la carpeta temporal, lo importa y lo ejecuta.

In [ ]:
pricing_source = """
...
"""

In [ ]:
pricing_path = write_module("pricing", pricing_source)
sys.modules.pop("pricing", None)  # forget any previous version of the module
captured = io.StringIO()
with contextlib.redirect_stdout(captured):  # yet another context manager: captures print()
    pricing = importlib.import_module("pricing")
assert captured.getvalue() == "", "Al importar no debe imprimirse nada"
assert pricing.with_vat(98.0) == 118.58
run = subprocess.run([sys.executable, pricing_path], capture_output=True, text=True)
assert run.stdout.strip() == "98.0 € + IVA = 118.58 €", run.stdout + run.stderr
print("✅ Correcto")

## 9. `async` y `await`

En las sesiones 5 y 6 el código de MCP es **asíncrono**. Basta con reconocer cuatro piezas:
- `async def` define una función asíncrona (una *corrutina*). Llamarla **no** la ejecuta: devuelve algo que hay que esperar.
- `await` espera a que termine. Solo puede escribirse dentro de otra función `async def`.
- `async with` es el `with` de la sección 7 para recursos asíncronos (por ejemplo, la conexión con un servidor MCP).
- En un script, `asyncio.run(main())` arranca la primera corrutina.

**En un notebook** ya hay un bucle asíncrono en marcha: `asyncio.run()` da error y se escribe `await` directamente
en la celda.

La ventaja: mientras una operación espera (a la red, a un modelo…), el programa puede atender otras.

In [ ]:
import asyncio
import time


async def check_stock(sku: str, seconds: float) -> str:
    await asyncio.sleep(seconds)  # simulates waiting for a network answer
    return f"{sku}: OK"


pending = check_stock("MOL-020", 0)
print(type(pending))  # a coroutine: nothing has run yet
print(await pending)

start = time.perf_counter()
print(await check_stock("CAF-001", 1))
print(await check_stock("CAF-003", 1))
print(f"Una detrás de otra: {time.perf_counter() - start:.1f} s")

start = time.perf_counter()
results = await asyncio.gather(check_stock("CAF-001", 1), check_stock("CAF-003", 1))
print(results, f"A la vez: {time.perf_counter() - start:.1f} s")

**Ejercicio 9.** Escribe `async def check_all(skus)` que compruebe **a la vez** todos los SKU con
`check_stock(sku, 0.5)` y devuelva la lista de resultados en el mismo orden.
Pista: `asyncio.gather(*corrutinas)` recibe las corrutinas separadas; el `*` desempaqueta una lista.

In [ ]:
async def check_all(skus: list[str]) -> list[str]:
    ...

In [ ]:
start = time.perf_counter()
results = await check_all(["CAF-001", "CAF-003", "MOL-020"])
elapsed = time.perf_counter() - start
assert list(results) == ["CAF-001: OK", "CAF-003: OK", "MOL-020: OK"]
assert elapsed < 1.2, f"Ha tardado {elapsed:.1f} s: ¿las has esperado una detrás de otra?"
print("✅ Correcto")

## 10. Leer un traceback

Cuando una excepción no se captura, Python imprime un **traceback**. Se lee **de abajo arriba**:
1. La **última línea**: el tipo de error y su mensaje.
2. Las líneas `File "...", line N, in ...`: el camino de llamadas, desde el script que ejecutaste (arriba) hasta la
   línea exacta que falló (abajo). `in <module>` es código suelto del fichero; si no, aparece el nombre de la función.

Con `traceback.print_exc()` podemos imprimirlo sin que se pare el notebook:

In [ ]:
import traceback


def order_line_price(sku: str) -> float:
    return find_product(sku)["price"]


try:
    order_line_price("XXX-999")
except KeyError:
    traceback.print_exc()

**Ejercicio 10.** Una alumna ejecuta un script suyo, `prueba_stock.py`, que usa `herramientas.py` del kit, y obtiene esto:

```
Traceback (most recent call last):
  File "/Users/ana/bloque4/prueba_stock.py", line 2, in <module>
    print(stock_producto("descafeinado"))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ana/bloque4/herramientas.py", line 28, in stock_producto
    return _consulta("SELECT sku, nombre, precio, stock FROM productos "
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ana/bloque4/herramientas.py", line 14, in _consulta
    return [dict(f) for f in con.execute(sql, params)]
                             ^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: no such table: productos
```

Rellena las variables:
- `error_type`: el tipo de error, tal como aparece.
- `failing_function`: la función en la que se produce el error.
- `fix`: la solución más probable (`"a"`, `"b"` o `"c"`):
  - a) instalar `sqlite3` con `pip install sqlite3`;
  - b) ejecutar `python crear_bd.py` en la carpeta `bloque4` (si `tostadero.db` no existe, `sqlite3.connect` crea un fichero vacío);
  - c) cambiar `productos` por `products` en la consulta.

In [ ]:
error_type = ...
failing_function = ...
fix = ...

In [ ]:
assert error_type.endswith("OperationalError")
assert failing_function == "_consulta"
assert fix == "b"
print("✅ Correcto")

## Reto final

Llega un pedido en JSON (como el que podría extraer un LLM de un correo). Escribe `order_total(order_json)` que:
1. Convierta el texto JSON en un diccionario.
2. Para cada línea, busque el producto con `find_product` y cree un `OrderLine` con su precio.
3. Devuelva el total del pedido redondeado a 2 decimales.
4. Si algún SKU no existe, deje que salga el `KeyError`.

In [ ]:
order_json = '''
{"customer": "Bar La Esquina",
 "lines": [{"sku": "CAF-001", "qty": 4}, {"sku": "CAF-003", "qty": 2}]}
'''

In [ ]:
def order_total(order_json: str) -> float:
    ...

In [ ]:
assert order_total(order_json) == 137.60
try:
    order_total('{"customer": "X", "lines": [{"sku": "NOPE", "qty": 1}]}')
    raise AssertionError("Debería haber lanzado KeyError")
except KeyError:
    pass
print("✅ Reto superado")